# 3/3 — Fine-tuning-specific checks: catastrophic forgetting + novel-class sanity

**This is a gate, not the real benchmark.** It answers one question fast, on Kaggle, with no new photography needed: *did this training run actually learn the new classes without wrecking the classes we kept?* The real product-relevant comparison — base YOLO vs. YOLOE vs. this fine-tuned model, all three scored on the same self-captured golden set — is `finetuning/evaluate.py`, run separately (usually locally, once the golden set exists — see `finetuning/README.md` Step 4 and `docs/FINE_TUNING_YOLO_GUIDE.md` Section 7). Don't skip straight to that: if this gate fails, there's no point capturing/annotating 300-400 golden-set photos yet.

**Prerequisite:** attach two inputs via **Add Data**:
- `02_train.ipynb`'s output (for `runs/lostfound_yolo26n/weights/best.pt`)
- `01_dataset_prep_and_eda.ipynb`'s output (for `lostfound/lostfound.yaml`'s own val split and `coco_forgetting_check/lostfound.yaml`)

In [ ]:
!pip install -q ultralytics==8.4.106

In [ ]:
from pathlib import Path

TRAIN_OUTPUT_DIR = Path("/kaggle/input/<notebook-02-train-slug>")  # <-- EDIT
PREP_OUTPUT_DIR = Path("/kaggle/input/<notebook-01-dataset-prep-slug>")  # <-- EDIT

BEST_PT = TRAIN_OUTPUT_DIR / "runs" / "lostfound_yolo26n" / "weights" / "best.pt"
TRAIN_VAL_YAML = PREP_OUTPUT_DIR / "lostfound" / "lostfound.yaml"
FORGETTING_YAML = PREP_OUTPUT_DIR / "coco_forgetting_check" / "lostfound.yaml"
for p in (BEST_PT, TRAIN_VAL_YAML, FORGETTING_YAML):
    assert p.exists(), f"{p} not found — check the input dataset slugs above."

## 1. Novel-class sanity check — on the training run's own val split

Confirms the 28 novel classes (indices 20-47) learned *something* — this is on data from the same distribution as training (LVIS/COCO), not the golden set, so treat it as a sanity check that training worked at all, not a real accuracy number for the product's actual use case.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(BEST_PT))
train_val_metrics = model.val(data=str(TRAIN_VAL_YAML), split="val", imgsz=640)
print("Training-set val -- mAP50:", train_val_metrics.box.map50, "mAP50-95:", train_val_metrics.box.map)
print("Per-class mAP50-95 for the 28 novel classes (indices 20-47):")
print(train_val_metrics.box.maps[20:48])

## 2. Catastrophic-forgetting check — on the COCO val2017-derived set

This is the actual fine-tuning-specific regression gate: did fine-tuning on the narrower 48-class scheme hurt the 20 kept COCO classes, scored on COCO's own (never-trained-on) `val2017` images? See `docs/FINE_TUNING_YOLO_GUIDE.md` Section 6 for why this check exists and is scoped to only the 20 kept classes, not all 80.

In [ ]:
forgetting_metrics = model.val(data=str(FORGETTING_YAML), split="val", imgsz=640)
print("COCO forgetting-check -- mAP50:", forgetting_metrics.box.map50, "mAP50-95:", forgetting_metrics.box.map)

## 3. Pass / fail

There's no universal numeric threshold here (no prior fine-tuned baseline to compare against yet) — use judgment:
- **Novel classes**: if several report `mAP50-95` near 0, re-check that class's instance count in `01_dataset_prep_and_eda.ipynb`'s EDA output before assuming the model failed — a near-zero-instance class can't have learned anything regardless of training quality.
- **Forgetting check**: compare against base YOLO26n's own published COCO mAP for those same 20 classes (run `YOLO("yolo26n.pt").val(data="coco.yaml")` for a reference, filtering to the 20 kept classes) — a small drop is expected and acceptable from narrowing `nc` 80→48 and partial-freezing; a collapse (near-zero on classes that used to work) means revisit `freeze=10` → `freeze=None`, or increase `--coco-sample-size` in `01_dataset_prep_and_eda.ipynb` and redo dataset prep.

**If this gate passes:** proceed to `finetuning/evaluate.py`'s real three-way golden-set benchmark (needs the self-captured, annotated golden set — `finetuning/README.md` Step 4). **If it doesn't:** fix the training run (more epochs, different freeze depth, more COCO sample images, or a Roboflow fallback for a sparse novel class) before spending time on golden-set photography/annotation.